In [ ]:
from market_simulation.models.order_batch_model import OrderBatchModel
from market_simulation.models.utils_order_batch_model import (
    OrderBatchTokenDataset, TokenSequenceSpec, collate_token_sequences, lm_loss_next_token
)
from torch.utils.data import DataLoader

# 1) dataset: each file holds tokens shaped (T,) or (N,T)
ds = OrderBatchTokenDataset(
    paths=["/path/to/tokens_0.npy", "/path/to/tokens_1.npz"],
    spec=TokenSequenceSpec(block_size=1024, stride=1024),  # e.g., 16 minutes * 64 tokens/min
)

dl = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_token_sequences)

# 2) model
model = OrderBatchModel(emb_dim=768, num_layers=12, num_heads=12, vocab_size=8192)

# 3) train step
for input_ids in dl:
    logits = model(input_ids)                 # (B,T,V)
    loss = lm_loss_next_token(logits, input_ids)
    loss.backward()
    break
